# Notebook 02 _ Prétraitement des données énergétiques

## Objectif

Ce notebook a pour objectif de préparer le jeu de données énergétique avant l'analyse exploratoire avancée et la modélisation.

Les principales étapes sont :

- recharger les données brutes ;
- créer une copie de travail ;
- supprimer les colonnes inutiles ;
- convertir et contrôler la variable temporelle ;
- vérifier la continuité chronologique ;
- analyser les valeurs négatives de production ;
- appliquer des règles de nettoyage traçables ;
- sauvegarder un jeu de données intermédiaire.

Les données brutes ne seront jamais modifiées directement.

In [1]:
# 1. IMPORTATION DES BIBLIOTHÈQUES

from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", "{:.3f}".format)

sns.set_theme(style="whitegrid")

print("Bibliothèques importées .")

Bibliothèques importées .


In [2]:
# 2. DÉFINITION DES CHEMINS

PROJECT_DIR = Path.cwd().parent

RAW_DATA_DIR = PROJECT_DIR / "data" / "raw"
INTERIM_DATA_DIR = PROJECT_DIR / "data" / "interim"
PROCESSED_DATA_DIR = PROJECT_DIR / "data" / "processed"
FIGURES_DIR = PROJECT_DIR / "figures"

INTERIM_DATA_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DATA_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

ENERGY_FILE = RAW_DATA_DIR / "Database.csv"

print("Projet :", PROJECT_DIR)
print("Fichier énergétique :", ENERGY_FILE)
print("Le fichier existe :", ENERGY_FILE.exists())

Projet : c:\Users\celes\Desktop\PFE
Fichier énergétique : c:\Users\celes\Desktop\PFE\data\raw\Database.csv
Le fichier existe : True


In [3]:
# 3. CHARGEMENT DES DONNÉES BRUTES

energy_raw = pd.read_csv(ENERGY_FILE)

# Création d'une copie de travail.
# energy_raw reste la version originale chargée depuis le fichier.
energy_clean = energy_raw.copy()

print("Dimensions des données brutes :", energy_raw.shape)
print("Dimensions de la copie :", energy_clean.shape)

Dimensions des données brutes : (315648, 13)
Dimensions de la copie : (315648, 13)


In [4]:
# 4. SUPPRESSION DES COLONNES TECHNIQUES INUTILES

columns_to_drop = [
    column
    for column in energy_clean.columns
    if column.lower().startswith("unnamed")
]

print("Colonnes à supprimer :", columns_to_drop)

energy_clean = energy_clean.drop(
    columns=columns_to_drop,
    errors="ignore"
)

print("Nouvelles dimensions :", energy_clean.shape)

Colonnes à supprimer : ['Unnamed: 0']
Nouvelles dimensions : (315648, 12)


In [5]:
# 5. CONVERSION DE LA VARIABLE TEMPORELLE

energy_clean["Time"] = pd.to_datetime(
    energy_clean["Time"],
    errors="coerce"
)

print("Type de Time :", energy_clean["Time"].dtype)
print("Dates non converties :", energy_clean["Time"].isna().sum())

display(energy_clean[["Time"]].head())

Type de Time : datetime64[us]
Dates non converties : 0


,Time
0,2019-01-01 00:00:00
1,2019-01-01 00:05:00
2,2019-01-01 00:10:00
3,2019-01-01 00:15:00
4,2019-01-01 00:20:00


In [6]:
# 6. CONTRÔLE DE LA CHRONOLOGIE

print("Date minimale :", energy_clean["Time"].min())
print("Date maximale :", energy_clean["Time"].max())

print(
    "La série est triée chronologiquement :",
    energy_clean["Time"].is_monotonic_increasing
)

print(
    "Nombre de dates dupliquées :",
    energy_clean["Time"].duplicated().sum()
)

Date minimale : 2019-01-01 00:00:00
Date maximale : 2021-12-31 23:55:00
La série est triée chronologiquement : True
Nombre de dates dupliquées : 0


In [7]:
# Calcul de l'écart entre deux observations successives
time_steps = energy_clean["Time"].diff().dropna()

print("Intervalles les plus fréquents :")
display(
    time_steps.value_counts()
    .head()
    .to_frame("Nombre d'occurrences")
)

unexpected_steps = time_steps[
    time_steps != pd.Timedelta(minutes=5)
]

print(
    "Nombre d'intervalles différents de 5 minutes :",
    len(unexpected_steps)
)

Intervalles les plus fréquents :


,Nombre d'occurrences
Time,
0 days 00:05:00,315647


Nombre d'intervalles différents de 5 minutes : 0


# 7. CONTRÔLE DE LA COHÉRENCE PHYSIQUE DES DONNÉES

Avant de construire un modèle d'intelligence artificielle, il est nécessaire de vérifier que les variables respectent les lois physiques. Cette étape permet de détecter d'éventuelles valeurs incohérentes qui pourraient dégrader les performances des modèles.

In [8]:
# 7. RECHERCHE DES VALEURS PHYSIQUEMENT IMPOSSIBLES

variables = [
    "DHI",
    "DNI",
    "GHI",
    "Wind_speed",
    "Humidity",
    "Temperature",
    "PV_production",
    "Wind_production",
    "Electric_demand"
]

for col in variables:

    print("=" * 50)
    print(col)

    print("Minimum :", energy_clean[col].min())
    print("Maximum :", energy_clean[col].max())

DHI
Minimum : 0.0
Maximum : 431.0
DNI
Minimum : 0.0
Maximum : 999.8
GHI
Minimum : 0.0
Maximum : 1058.2
Wind_speed
Minimum : 0.66
Maximum : 8.540000000000001
Humidity
Minimum : 11.572
Maximum : 88.688
Temperature
Minimum : -0.5400000000000006
Maximum : 39.02
PV_production
Minimum : -145
Maximum : 13191
Wind_production
Minimum : -2476
Maximum : 5743
Electric_demand
Minimum : 14662
Maximum : 47067


## Résumé du contrôle de cohérence physique

Les variables météorologiques présentent des bornes globalement plausibles :

- les rayonnements DHI, DNI et GHI sont toujours positifs ou nuls ;
- la vitesse du vent est toujours positive ;
- l'humidité est comprise entre 11,57 % et 88,69 % ;
- la température varie entre -0,54 °C et 39,02 °C ;
- la demande électrique est toujours positive.

En revanche, des valeurs négatives sont observées dans les variables
`PV_production` et `Wind_production`.

Ces observations ne seront pas corrigées automatiquement, car leur origine
doit d'abord être étudiée. Elles peuvent notamment correspondre à une puissance
nette, à une consommation auxiliaire, à un bruit de mesure ou à une convention
propre à la source des données.

In [9]:
# 8. ANALYSE DES PRODUCTIONS NÉGATIVES

negative_production_summary = pd.DataFrame({
    "Variable": ["PV_production", "Wind_production"],
    "Nombre de valeurs négatives": [
        (energy_clean["PV_production"] < 0).sum(),
        (energy_clean["Wind_production"] < 0).sum()
    ],
    "Pourcentage (%)": [
        (energy_clean["PV_production"] < 0).mean() * 100,
        (energy_clean["Wind_production"] < 0).mean() * 100
    ],
    "Minimum": [
        energy_clean["PV_production"].min(),
        energy_clean["Wind_production"].min()
    ]
})

negative_production_summary["Pourcentage (%)"] = (
    negative_production_summary["Pourcentage (%)"].round(3)
)

negative_production_summary

,Variable,Nombre de valeurs négatives,Pourcentage (%),Minimum
0,PV_production,112829,35.745,-145
1,Wind_production,216,0.068,-2476


In [10]:
# Vérification des valeurs photovoltaïques négatives
# selon la présence ou l'absence de rayonnement solaire.

pv_negative_analysis = pd.DataFrame({
    "Situation": [
        "PV négative avec GHI nul",
        "PV négative avec GHI positif"
    ],
    "Nombre": [
        (
            (energy_clean["PV_production"] < 0)
            & (energy_clean["GHI"] == 0)
        ).sum(),
        (
            (energy_clean["PV_production"] < 0)
            & (energy_clean["GHI"] > 0)
        ).sum()
    ]
})

pv_negative_analysis

,Situation,Nombre
0,PV négative avec GHI nul,106122
1,PV négative avec GHI positif,6707


# 9. CRÉATION DES VARIABLES TEMPORELLES

Afin de faciliter l'apprentissage des modèles de Machine Learning, nous extrayons plusieurs informations de la variable temporelle :

- année
- mois
- jour
- heure
- minute
- jour de la semaine
- trimestre
- week-end

Ces variables permettront aux modèles de capturer les comportements saisonniers et journaliers de la production énergétique.

In [11]:
# EXTRACTION DES VARIABLES TEMPORELLES

energy_clean["Year"] = energy_clean["Time"].dt.year

energy_clean["Month"] = energy_clean["Time"].dt.month

energy_clean["Day"] = energy_clean["Time"].dt.day

energy_clean["Hour"] = energy_clean["Time"].dt.hour

energy_clean["Minute"] = energy_clean["Time"].dt.minute

energy_clean["Weekday"] = energy_clean["Time"].dt.dayofweek

energy_clean["Quarter"] = energy_clean["Time"].dt.quarter

energy_clean["Is_weekend"] = (
    energy_clean["Weekday"] >= 5
).astype(int)

energy_clean.head()

,Time,Season,Day_of_the_week,DHI,DNI,GHI,Wind_speed,Humidity,Temperature,PV_production,Wind_production,Electric_demand,Year,Month,Day,Hour,Minute,Weekday,Quarter,Is_weekend
0,2019-01-01 00:00:00,1,1,0.000,0.000,0.000,2.880,56.036,1.820,0,2810,22216,2019,1,1,0,0,1,1,0
1,2019-01-01 00:05:00,1,1,0.000,0.000,0.000,2.880,56.036,1.820,0,2862,22106,2019,1,1,0,5,1,1,0
2,2019-01-01 00:10:00,1,1,0.000,0.000,0.000,2.880,56.194,1.780,0,2916,22130,2019,1,1,0,10,1,1,0
3,2019-01-01 00:15:00,1,1,0.000,0.000,0.000,2.880,56.344,1.740,0,2920,22040,2019,1,1,0,15,1,1,0
4,2019-01-01 00:20:00,1,1,0.000,0.000,0.000,2.840,56.440,1.720,0,2902,21963,2019,1,1,0,20,1,1,0


In [12]:
print(energy_clean[
    [
        "Time",
        "Year",
        "Month",
        "Day",
        "Hour",
        "Minute",
        "Weekday",
        "Quarter",
        "Is_weekend"
    ]
].head())

                 Time  Year  Month  Day  Hour  Minute  Weekday  Quarter  \
0 2019-01-01 00:00:00  2019      1    1     0       0        1        1   
1 2019-01-01 00:05:00  2019      1    1     0       5        1        1   
2 2019-01-01 00:10:00  2019      1    1     0      10        1        1   
3 2019-01-01 00:15:00  2019      1    1     0      15        1        1   
4 2019-01-01 00:20:00  2019      1    1     0      20        1        1   

   Is_weekend  
0           0  
1           0  
2           0  
3           0  
4           0  


In [13]:
# 10. SAUVEGARDE DU JEU DE DONNÉES PRÉTRAITÉ

output_file = PROCESSED_DATA_DIR / "energy_preprocessed.csv"

energy_clean.to_csv(
    output_file,
    index=False
)

print(" Dataset prétraité.")
print(" Emplacement :", output_file)
print(" Dimensions :", energy_clean.shape)

 Dataset prétraité.
 Emplacement : c:\Users\celes\Desktop\PFE\data\processed\energy_preprocessed.csv
 Dimensions : (315648, 20)


# Conclusion du Notebook 02

Le jeu de données énergétique a été préparé sans modifier le fichier brut.

Les opérations réalisées comprennent :

- la suppression de la colonne technique `Unnamed: 0` ;
- la conversion de la variable `Time` au format temporel ;
- la vérification de la continuité de la série à une fréquence de cinq minutes ;
- le contrôle des doublons et des dates non converties ;
- l'analyse des bornes physiques des variables ;
- l'identification de productions photovoltaïques et éoliennes négatives ;
- la création de premières variables calendaires ;
- la sauvegarde d'un fichier intermédiaire reproductible.

Les productions négatives sont conservées à ce stade. Leur traitement sera
déterminé après une analyse de leur répartition temporelle et de leur relation
avec les variables météorologiques.